In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install wandb -q

In [3]:
import numpy as np
import wandb
from wandb.integration.keras import WandbMetricsLogger
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json, os
from google.colab import userdata

print(f"TF version: {tf.__version__}")

TF version: 2.20.0


In [4]:
from google.colab import userdata
import wandb

wandb_api_key = userdata.get('WANDB_API_KEY_1')
wandb.login(key=wandb_api_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akeke23 (akeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

LabelEncoding


In [6]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

data = np.load('/content/drive/MyDrive/Ketastasia/data/dataset_seq15_ready.npz')

X_train = data['X_train']
y_train_text = data['y_train']
X_val   = data['X_val']
y_val_text   = data['y_val']


# 2. ტექსტური ლეიბლების ციფრებში გადაყვანა
le = LabelEncoder()
y_train_idx = le.fit_transform(y_train_text)
y_val_idx   = le.transform(y_val_text)

classes = list(le.classes_)
n_classes = len(classes)
n_timesteps = X_train.shape[1]   # 15 კადრი
n_features  = X_train.shape[2]   # 29 ფიჩერი

# 3. გადაყვანა One-Hot Encoding-ში (რადგან ბოლო ფენა Softmax-ია)
y_train = to_categorical(y_train_idx, num_classes=n_classes)
y_val   = to_categorical(y_val_idx, num_classes=n_classes)

print("=== 🎉 მონაცემები სრულად მომზადდა! ===")
print(f"Train სეტი: X={X_train.shape}, y={y_train.shape}")
print(f"Val სეტი:   X={X_val.shape},   y={y_val.shape}")
print(f"კლასების რაოდენობა: {n_classes} -> {classes}")

=== 🎉 მონაცემები სრულად მომზადდა! ===
Train სეტი: X=(2940, 15, 29), y=(2940, 25)
Val სეტი:   X=(580, 15, 29),   y=(580, 25)
კლასების რაოდენობა: 25 -> [np.str_('bench_press'), np.str_('bicep_curl'), np.str_('chest_fly'), np.str_('clean_and_jerk'), np.str_('deadlift'), np.str_('decline_bench_press'), np.str_('hammer_curl'), np.str_('hip_thrust'), np.str_('incline_bench_press'), np.str_('jump_rope'), np.str_('lat_pulldown'), np.str_('lateral_raise'), np.str_('leg_extension'), np.str_('leg_raises'), np.str_('plank'), np.str_('pullup'), np.str_('pushup'), np.str_('romanian_deadlift'), np.str_('russian_twist'), np.str_('shoulder_press'), np.str_('situp'), np.str_('squat'), np.str_('t_bar_row'), np.str_('tricep_dips'), np.str_('tricep_pushdown')]


In [7]:
# =====================================================================
# Cell 5: LSTM არქიტექტურა (Baseline - Dropout-ის გარეშე)
# =====================================================================
def build_lstm_baseline(hidden=64, n_layers=1, lr=0.001):
    model = keras.Sequential()

    # შესავალი ფენა (15 კადრი, 29 ფიჩერი)
    model.add(layers.Input(shape=(n_timesteps, n_features)))

    # მრავალშრიანი LSTM (Dropout-ების გარეშე)
    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        model.add(layers.LSTM(hidden, return_sequences=return_seq))

    # გადაწყვეტილების მიმღები Dense ფენა (აქაც Dropout-ის გარეშე)
    model.add(layers.Dense(64, activation='relu'))

    # გამომავალი შრე - Softmax კლასიფიკატორი
    model.add(layers.Dense(n_classes, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [8]:
# =====================================================================
# Cell 7: ექსპერიმენტის კონფიგურაციების განსაზღვრა (Baseline ვერსია)
# =====================================================================
configs = [
    {
        'hidden': 64, 'n_layers': 1, 'dropout': 0.0,
        'lr': 0.01, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 64, 'n_layers': 1, 'dropout': 0.0,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 128, 'n_layers': 2, 'dropout': 0.0,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 128, 'n_layers': 2, 'dropout': 0.0,
        'lr': 0.0005, 'batch_size': 64, 'epochs': 40
    },
    {
        'hidden': 256, 'n_layers': 2, 'dropout': 0.0,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    }
]
print(f"მომზადებულია {len(configs)} სხვადასხვა ბეისლაინ ექსპერიმენტი გასაშვებად! 🚀")

მომზადებულია 5 სხვადასხვა ბეისლაინ ექსპერიმენტი გასაშვებად! 🚀


In [10]:
# =====================================================================
# Cell 8: ექსპერიმენტების ავტომატური გაშვება (არადროპაუტი არა კლასების წონა)
# =====================================================================
best_val_acc = 0
best_run_name = None

# საქაღალდე მოდელების შესანახად
os.makedirs('/content/drive/MyDrive/Ketastasia/models', exist_ok=True)

for cfg in configs:
    # სახელი თითოეული რანისთვის
    run_name = f"lstm-h{cfg['hidden']}-l{cfg['n_layers']}-lr{cfg['lr']}-bs{cfg['batch_size']}-baseline"

    print(f"\n{'-'*60}\n🚀 იწყება Baseline ექსპერიმენტი: {run_name}\n{'-'*60}")

    # 1. WandB-ს ინიციალიზაცია
    wandb.init(
        project='ildolcefarniente',
        group='p1_lstm',
        name=run_name,
        config={
            **cfg,
            'dropout_included': False,
            'class_weights_included': False,
            'n_timesteps': n_timesteps,
            'n_features': n_features,
            'n_classes': n_classes,
            'pipeline': 'pipeline1A_13kp',
            'normalization': 'per_frame_hip_centered'
        },
        reinit=True
    )

    # 2. მოდელის აწყობა -> იძახებს ბეისლაინ ფუნქციას (დროპაუტის გარეშე)
    model = build_lstm_baseline(
        hidden=cfg['hidden'],
        n_layers=cfg['n_layers'],
        lr=cfg['lr']
    )

    # 3. ქოლბექები (მეტრიკების ლოგერი, ჭკვიანი გაჩერება და ლერნინგ რეიტის ვარდნა)
    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=4,
            verbose=1
        )
    ]

    # 4. მოდელის წვრთნა (Training) -> 'history =' წავშალეთ, პირდაპირ ვავარჯიშებთ
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        verbose=1
    )

    # 5. საბოლოო შეფასება (მხოლოდ Validation სეტზე!)
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    # პრედიქციები ვალიდაციის მონაცემებზე რეპორტისა და მატრიცისთვის
    y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)
    y_true_val = np.argmax(y_val, axis=1)

    # კლასიფიკაციის რეპორტი -> 👇 აქ ჩავამატეთ labels პარამეტრი, რომ არ გაფრინდეს!
    report_val = classification_report(
        y_true_val,
        y_pred_val,
        labels=list(range(len(classes))),
        target_names=classes,
        output_dict=True
    )

    # 6. Confusion Matrix გრაფიკის აგება (Validation)
    cm_val = confusion_matrix(y_true_val, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Val Confusion Matrix (Baseline) — {run_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Val Label')
    ax.set_xlabel('Predicted Val Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'/content/{run_name}_val_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    # 7. ფინალური მეტრიკების ატვირთვა WandB-ზე
    wandb.log({
        'final_val_accuracy':        val_acc,
        'final_val_loss':            val_loss,
        'val_f1_macro':              report_val['macro avg']['f1-score'],
        'val_f1_weighted':           report_val['weighted avg']['f1-score'],
        'val_confusion_matrix_db':   wandb.plot.confusion_matrix(
            probs=None, y_true=y_true_val.tolist(), preds=y_pred_val.tolist(), class_names=classes
        ),
        'val_confusion_matrix_img':  wandb.Image(cm_path)
    })

    # 8. მოდელის შენახვა Google Drive-ზე (.h5 ფორმატში)
    model_path = f'/content/drive/MyDrive/Ketastasia/models/{run_name}.h5'
    model.save(model_path)

    # 9. თვალყურის დევნება საუკეთესო ვალიდაციის მოდელზე
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_run_name = run_name

    print(f"✨ Run {run_name} დასრულდა. Val Accuracy: {val_acc:.4f}")
    wandb.finish()

print(f"\n{'='*60}\n🏆 ბეისლაინები დასრულდა! საუკეთესო მოდელი ვალიდაციით: {best_run_name} (Val Acc: {best_val_acc:.4f})\n{'='*60}")


------------------------------------------------------------
🚀 იწყება Baseline ექსპერიმენტი: lstm-h64-l1-lr0.01-bs32-baseline
------------------------------------------------------------


epoch/accuracy,▁▂▂▃▄▄▄▅▆▆▆▆▇▇▇▇████████
epoch/epoch,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇██
epoch/learning_rate,████████▄▄▄▄▂▂▂▂▁▁▁▁▁▁▁▁
epoch/loss,█▇▆▆▅▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▂▅▄▅▅▆▇▆▅▇█▇▇███▇█▇▇▇▇
epoch/val_loss,▂▁▂▁▂▁▂▂▃▂▄▄▄▅▅▅▆▆▇▇▇███
epoch/accuracy,0.9517
epoch/epoch,23
epoch/learning_rate,0.00062
epoch/loss,0.17363
epoch/val_accuracy,0.43448


Epoch 1/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.3126 - loss: 2.3200 - val_accuracy: 0.2638 - val_loss: 2.5107 - learning_rate: 0.0100
Epoch 2/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3714 - loss: 1.9646 - val_accuracy: 0.3121 - val_loss: 2.4245 - learning_rate: 0.0100
Epoch 3/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4391 - loss: 1.7554 - val_accuracy: 0.3362 - val_loss: 2.3145 - learning_rate: 0.0100
Epoch 4/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.4942 - loss: 1.5709 - val_accuracy: 0.3397 - val_loss: 2.4374 - learning_rate: 0.0100
Epoch 5/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.5289 - loss: 1.4599 - val_accuracy: 0.3586 - val_loss: 2.4218 - learning_rate: 0.0100
Epoch 6/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5551 - loss: 1.3404 - val_accuracy: 0.3897 - val_loss: 2.3585 - learning_rate: 0.0100
Epoch 7/40
90/92 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5731 - loss: 1.2375
Epoch 7: 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✨ Run lstm-h64-l1-lr0.01-bs32-baseline დასრულდა. Val Accuracy: 0.4603


...

Epoch 1/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 20s 171ms/step - accuracy: 0.2983 - loss: 2.4525 - val_accuracy: 0.2724 - val_loss: 2.5204 - learning_rate: 0.0010
Epoch 2/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 165ms/step - accuracy: 0.3752 - loss: 2.0587 - val_accuracy: 0.3000 - val_loss: 2.4712 - learning_rate: 0.0010
Epoch 3/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 156ms/step - accuracy: 0.4139 - loss: 1.8999 - val_accuracy: 0.3207 - val_loss: 2.3975 - learning_rate: 0.0010
Epoch 4/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 146ms/step - accuracy: 0.4337 - loss: 1.7810 - val_accuracy: 0.3448 - val_loss: 2.2968 - learning_rate: 0.0010
Epoch 5/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 152ms/step - accuracy: 0.4731 - loss: 1.6593 - val_accuracy: 0.3362 - val_loss: 2.3932 - learning_rate: 0.0010
Epoch 6/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 160ms/step - accuracy: 0.4922 - loss: 1.5601 - val_accuracy: 0.3759 - val_loss: 2.3501 - learning_rate: 0.0010
Epoch 7/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 19s 145ms/step - accuracy: 0.5299 - loss: 1.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

✨ Run lstm-h256-l2-lr0.001-bs32-baseline დასრულდა. Val Accuracy: 0.4603


epoch/accuracy,▁▂▂▂▃▃▃▄▄▄▄▅▆▆▆▆▇▇▇▇████████████████████
epoch/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch/learning_rate,████████████▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▇▆▆▆▅▅▅▄▄▄▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▃▄▃▅▅▅▅▅▅▆▇▆▇▆▇▇▇▇▇▇█▇███▇████████████
epoch/val_loss,▃▃▂▁▂▂▁▁▁▁▂▃▂▃▅▄▅▅▅▆▆▆▇▇▇▇▇█████████████
final_val_accuracy,▁
final_val_loss,▁
val_f1_macro,▁
val_f1_weighted,▁
epoch/accuracy,0.96259



🏆 ბეისლაინები დასრულდა! საუკეთესო მოდელი ვალიდაციით: lstm-h64-l1-lr0.01-bs32-baseline (Val Acc: 0.4603)


add dropout

In [11]:
# =====================================================================
# Cell 5 (V2): LSTM არქიტექტურა (Dropout-ის რეგულარიზაციით)
# =====================================================================
def build_lstm_with_dropout(hidden=64, n_layers=1, dropout_rate=0.3, lr=0.001):
    model = keras.Sequential()

    # შესავალი ფენა (15 კადრი, 29 ფიჩერი)
    model.add(layers.Input(shape=(n_timesteps, n_features)))

    # მრავალშრიანი LSTM დროპაუტებით
    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        model.add(layers.LSTM(
            hidden,
            return_sequences=return_seq,
            dropout=dropout_rate,           # 👈 დროპაუტი ინფუთებზე
            recurrent_dropout=dropout_rate  # 👈 დროპაუტი შიდა მეხსიერების კავშირებზე
        ))

    # გადაწყვეტილების მიმღები Dense ფენა
    model.add(layers.Dense(64, activation='relu'))

    # ცალკე დროპაუტი კლასიფიკატორის წინ
    model.add(layers.Dropout(dropout_rate))

    # გამომავალი შრე - Softmax კლასიფიკატორი
    model.add(layers.Dense(n_classes, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [12]:
# =====================================================================
# Cell 7: ექსპერიმენტის კონფიგურაციების განსაზღვრა (Baseline ვერსია)
# =====================================================================
configs = [
    {
        'hidden': 64, 'n_layers': 1, 'dropout': 0.3,
        'lr': 0.01, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 64, 'n_layers': 1, 'dropout': 0.3,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 128, 'n_layers': 2, 'dropout': 0.4,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 128, 'n_layers': 2, 'dropout': 0.4,
        'lr': 0.0005, 'batch_size': 64, 'epochs': 40
    },
    {
        'hidden': 256, 'n_layers': 2, 'dropout': 0.3,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    }
]
print(f"მომზადებულია {len(configs)} სხვადასხვა ბეისლაინ ექსპერიმენტი გასაშვებად! 🚀")

მომზადებულია 5 სხვადასხვა ბეისლაინ ექსპერიმენტი გასაშვებად! 🚀


In [13]:
best_val_acc = 0
best_run_name = None

# საქაღალდე მოდელების შესანახად
os.makedirs('/content/drive/MyDrive/Ketastasia/models', exist_ok=True)

for cfg in configs:
    # სახელი თითოეული რანისთვის (ბოლოში ეწერება -dropout)
    run_name = f"lstm-h{cfg['hidden']}-l{cfg['n_layers']}-d{cfg['dropout']}-lr{cfg['lr']}-bs{cfg['batch_size']}-dropout"

    print(f"\n{'-'*60}\n🚀 იწყება Dropout ექსპერიმენტი: {run_name}\n{'-'*60}")

    # 1. WandB-ს ინიციალიზაცია
    wandb.init(
        project='ildolcefarniente',
        group='p1_lstm',
        name=run_name,
        config={
            **cfg,
            'dropout_included': True,        # 👈 გასწორდა True (დიდი ასოთი)
            'class_weights_included': False,
            'n_timesteps': n_timesteps,
            'n_features': n_features,
            'n_classes': n_classes,
            'pipeline': 'pipeline1A_13kp',
            'normalization': 'per_frame_hip_centered'
        },
        reinit=True
    )

    # 2. მოდელის აწყობა -> ვიძახებთ Dropout-იან ფუნქციას და ვაყოლებთ 'dropout_rate'-ს!
    model = build_lstm_with_dropout(
        hidden=cfg['hidden'],
        n_layers=cfg['n_layers'],
        dropout_rate=cfg['dropout'],         # 👈 კონფიგიდან იღებს მნიშვნელობას
        lr=cfg['lr']
    )

    # 3. ქოლბექები (მეტრიკების ლოგერი, ჭკვიანი გაჩერება და ლერნინგ რეიტის ვარდნა)
    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            payout_rate=None, # (აქ უბრალოდ წინა პარამეტრებია)
            patience=4,
            verbose=1
        )
    ]

    # 4. მოდელის წვრთნა (Training)
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        verbose=1
    )

    # 5. საბოლოო შეფასება (მხოლოდ Validation სეტზე!)
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    # პრედიქციები ვალიდაციის მონაცემებზე რეპორტისა და მატრიცისთვის
    y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)
    y_true_val = np.argmax(y_val, axis=1)

    # კლასიფიკაციის რეპორტი
    report_val = classification_report(
        y_true_val,
        y_pred_val,
        labels=list(range(len(classes))),
        target_names=classes,
        output_dict=True
    )

    # 6. Confusion Matrix გრაფიკის აგება (Validation)
    cm_val = confusion_matrix(y_true_val, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Val Confusion Matrix (Dropout) — {run_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Val Label')
    ax.set_xlabel('Predicted Val Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'/content/{run_name}_val_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    # 7. ფინალური მეტრიკების ატვირთვა WandB-ზე
    wandb.log({
        'final_val_accuracy':        val_acc,
        'final_val_loss':            val_loss,
        'val_f1_macro':              report_val['macro avg']['f1-score'],
        'val_f1_weighted':           report_val['weighted avg']['f1-score'],
        'val_confusion_matrix_db':   wandb.plot.confusion_matrix(
            probs=None, y_true=y_true_val.tolist(), preds=y_pred_val.tolist(), class_names=classes
        ),
        'val_confusion_matrix_img':  wandb.Image(cm_path)
    })

    # 8. მოდელის შენახვა Google Drive-ზე (.h5 ფორმატში)
    model_path = f'/content/drive/MyDrive/Ketastasia/models/{run_name}.h5'
    model.save(model_path)

    # 9. თვალყურის დევნება საუკეთესო ვალიდაციის მოდელზე
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_run_name = run_name

    print(f"✨ Run {run_name} დასრულდა. Val Accuracy: {val_acc:.4f}")
    wandb.finish()

print(f"\n{'='*60}\n🏆 Dropout ექსპერიმენტები დასრულდა! საუკეთესო მოდელი ვალიდაციით: {best_run_name} (Val Acc: {best_val_acc:.4f})\n{'='*60}")


------------------------------------------------------------
🚀 იწყება Dropout ექსპერიმენტი: lstm-h64-l1-d0.3-lr0.01-bs32-dropout
------------------------------------------------------------


Epoch 1/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.2255 - loss: 2.6761 - val_accuracy: 0.1966 - val_loss: 2.5764 - learning_rate: 0.0100
Epoch 2/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.2837 - loss: 2.3946 - val_accuracy: 0.2569 - val_loss: 2.5101 - learning_rate: 0.0100
Epoch 3/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.3007 - loss: 2.2883 - val_accuracy: 0.2810 - val_loss: 2.4275 - learning_rate: 0.0100
Epoch 4/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.3422 - loss: 2.1757 - val_accuracy: 0.2931 - val_loss: 2.4240 - learning_rate: 0.0100
Epoch 5/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.3558 - loss: 2.1227 - val_accuracy: 0.2914 - val_loss: 2.3835 - learning_rate: 0.0100
Epoch 6/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.3616 - loss: 2.0798 - val_accuracy: 0.3000 - val_loss: 2.3764 - learning_rate: 0.0100
Epoch 7/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.3680 - loss: 2.0601 - val_acc

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

✨ Run lstm-h64-l1-d0.3-lr0.01-bs32-dropout დასრულდა. Val Accuracy: 0.3983


epoch/accuracy,▁▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇█▇██████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,███████████████████▄▄▄▄▂▂▂▂▁▁▁
epoch/loss,█▆▆▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
epoch/val_accuracy,▁▃▄▄▄▅▆▅▇▅▅▆▆▆▇▆▇▆▆▇██▇█▇▇▇▇▇█
epoch/val_loss,█▇▅▅▄▄▄▄▃▃▄▄▁▄▁▂▃▂▂▂▂▂▃▂▃▃▃▂▂▂
final_val_accuracy,▁
final_val_loss,▁
val_f1_macro,▁
val_f1_weighted,▁
epoch/accuracy,0.48571


...

Epoch 1/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 42s 357ms/step - accuracy: 0.1918 - loss: 2.7950 - val_accuracy: 0.2690 - val_loss: 2.5159 - learning_rate: 0.0010
Epoch 2/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 29s 315ms/step - accuracy: 0.2388 - loss: 2.5857 - val_accuracy: 0.2672 - val_loss: 2.4556 - learning_rate: 0.0010
Epoch 3/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 41s 320ms/step - accuracy: 0.2793 - loss: 2.4866 - val_accuracy: 0.2862 - val_loss: 2.4263 - learning_rate: 0.0010
Epoch 4/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 30s 324ms/step - accuracy: 0.2881 - loss: 2.4077 - val_accuracy: 0.2707 - val_loss: 2.4199 - learning_rate: 0.0010
Epoch 5/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 30s 325ms/step - accuracy: 0.3126 - loss: 2.3121 - val_accuracy: 0.2552 - val_loss: 2.4102 - learning_rate: 0.0010
Epoch 6/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 30s 327ms/step - accuracy: 0.3323 - loss: 2.2329 - val_accuracy: 0.2879 - val_loss: 2.3522 - learning_rate: 0.0010
Epoch 7/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 29s 314ms/step - accuracy: 0.3500 - loss: 2.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

✨ Run lstm-h256-l2-d0.3-lr0.001-bs32-dropout დასრულდა. Val Accuracy: 0.4052


epoch/accuracy,▁▂▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇███████████
epoch/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch/learning_rate,█████████████████████▄▄▄▄▄▄▄▂▂▂▂▁▁▁▁▁▁▁▁
epoch/loss,█▇▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▂▂▂▂▁▃▄▄▄▅▅▅▅▆▆▅▆▆▆▇▇▇▇▇▇▇▇▇███▇█▇██████
epoch/val_loss,█▇▆▆▆▄▃▅▄▃▃▂▄▂▂▄▁▂▃▂▂▁▃▁▃▁▁▂▂▁▂▂▃▃▃▃▃▃▃▃
final_val_accuracy,▁
final_val_loss,▁
val_f1_macro,▁
val_f1_weighted,▁
epoch/accuracy,0.55714



🏆 Dropout ექსპერიმენტები დასრულდა! საუკეთესო მოდელი ვალიდაციით: lstm-h256-l2-d0.3-lr0.001-bs32-dropout (Val Acc: 0.4052)


added class weight

In [14]:
# =====================================================================
# კლასების უთანაბრობის დაბალანსება (Class Weights)
# =====================================================================
from sklearn.utils.class_weight import compute_class_weight

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(n_classes),
    y=y_train_idx
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}

print("კლასების წონები (Class Weights):")
for i, cls in enumerate(classes):
    print(f"  {i} {cls}: {class_weight_dict[i]:.3f}")

კლასების წონები (Class Weights):
  0 bench_press: 1.050
  1 bicep_curl: 1.527
  2 chest_fly: 1.993
  3 clean_and_jerk: 0.305
  4 deadlift: 2.178
  5 decline_bench_press: 2.306
  6 hammer_curl: 1.265
  7 hip_thrust: 1.656
  8 incline_bench_press: 1.176
  9 jump_rope: 39.200
  10 lat_pulldown: 1.611
  11 lateral_raise: 0.912
  12 leg_extension: 1.321
  13 leg_raises: 1.188
  14 plank: 0.490
  15 pullup: 0.721
  16 pushup: 0.805
  17 romanian_deadlift: 1.384
  18 russian_twist: 1.352
  19 shoulder_press: 1.367
  20 situp: 1.867
  21 squat: 0.340
  22 t_bar_row: 0.891
  23 tricep_dips: 0.926
  24 tricep_pushdown: 1.704


In [15]:
# =====================================================================
# Cell 5: LSTM არქიტექტურა (Baseline - Dropout-ის გარეშე)
# =====================================================================
def build_lstm_baseline(hidden=64, n_layers=1, lr=0.001):
    model = keras.Sequential()

    # შესავალი ფენა (15 კადრი, 29 ფიჩერი)
    model.add(layers.Input(shape=(n_timesteps, n_features)))

    # მრავალშრიანი LSTM (Dropout-ების გარეშე)
    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        model.add(layers.LSTM(hidden, return_sequences=return_seq))

    # გადაწყვეტილების მიმღები Dense ფენა (აქაც Dropout-ის გარეშე)
    model.add(layers.Dense(64, activation='relu'))

    # გამომავალი შრე - Softmax კლასიფიკატორი
    model.add(layers.Dense(n_classes, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [16]:
# =====================================================================
# Cell 7: ექსპერიმენტის კონფიგურაციების განსაზღვრა (Class Weights ვერსია)
# =====================================================================
configs = [
    {
        'hidden': 64, 'n_layers': 1,
        'lr': 0.01, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 64, 'n_layers': 1,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 128, 'n_layers': 2,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 128, 'n_layers': 2,
        'lr': 0.0005, 'batch_size': 64, 'epochs': 40
    },
    {
        'hidden': 256, 'n_layers': 2,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    }
]
print(f"მომზადებულია {len(configs)} სხვადასხვა Class Weights ექსპერიმენტი გასაშვებად! ⚖️")

მომზადებულია 5 სხვადასხვა Class Weights ექსპერიმენტი გასაშვებად! ⚖️


In [17]:
# =====================================================================
# Cell 8: ექსპერიმენტების ავტომატური გაშვება (მხოლოდ Class Weights)
# =====================================================================
best_val_acc = 0
best_run_name = None

# საქაღალდე მოდელების შესანახად
os.makedirs('/content/drive/MyDrive/Ketastasia/models', exist_ok=True)

for cfg in configs:
    # სახელი თითოეული რანისთვის (ბოლოში ეწერება -classweights)
    run_name = f"lstm-h{cfg['hidden']}-l{cfg['n_layers']}-lr{cfg['lr']}-bs{cfg['batch_size']}-classweights"

    print(f"\n{'-'*60}\n🚀 იწყება Class Weights ექსპერიმენტი: {run_name}\n{'-'*60}")

    # 1. WandB-ს ინიციალიზაცია
    wandb.init(
        project='ildolcefarniente',
        group='p1_lstm',
        name=run_name,
        config={
            **cfg,
            'dropout_included': False,
            'class_weights_included': True,   # 👈 ახლა ეს გახდა True
            'n_timesteps': n_timesteps,
            'n_features': n_features,
            'n_classes': n_classes,
            'pipeline': 'pipeline1A_13kp',
            'normalization': 'per_frame_hip_centered'
        },
        reinit=True
    )

    # 2. მოდელის აწყობა -> ვაბრუნებთ ბეისლაინ ფუნქციას (დროპაუტის გარეშე)
    model = build_lstm_baseline(
        hidden=cfg['hidden'],
        n_layers=cfg['n_layers'],
        lr=cfg['lr']
    )

    # 3. ქოლბექები (მეტრიკების ლოგერი, ჭკვიანი გაჩერება და ლერნინგ რეიტის ვარდნა)
    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=4,
            verbose=1
        )
    ]

    # 4. მოდელის წვრთნა -> ჩავამატეთ class_weight პარამეტრი!
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        class_weight=class_weight_dict,       # 👈 აი ეს მნიშვნელოვანი ხაზი!
        callbacks=callbacks,
        verbose=1
    )

    # 5. საბოლოო შეფასება (მხოლოდ Validation სეტზე!)
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    # პრედიქციები ვალიდაციის მონაცემებზე რეპორტისა და მატრიცისთვის
    y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)
    y_true_val = np.argmax(y_val, axis=1)

    # კლასიფიკაციის რეპორტი
    report_val = classification_report(
        y_true_val,
        y_pred_val,
        labels=list(range(len(classes))),
        target_names=classes,
        output_dict=True
    )

    # 6. Confusion Matrix გრაფიკის აგება (Validation)
    cm_val = confusion_matrix(y_true_val, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Val Confusion Matrix (Class Weights) — {run_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Val Label')
    ax.set_xlabel('Predicted Val Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'/content/{run_name}_val_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    # 7. ფინალური მეტრიკების ატვირთვა WandB-ზე
    wandb.log({
        'final_val_accuracy':        val_acc,
        'final_val_loss':            val_loss,
        'val_f1_macro':              report_val['macro avg']['f1-score'],
        'val_f1_weighted':           report_val['weighted avg']['f1-score'],
        'val_confusion_matrix_db':   wandb.plot.confusion_matrix(
            probs=None, y_true=y_true_val.tolist(), preds=y_pred_val.tolist(), class_names=classes
        ),
        'val_confusion_matrix_img':  wandb.Image(cm_path)
    })

    # 8. მოდელის შენახვა Google Drive-ზე (.h5 ფორმატში)
    model_path = f'/content/drive/MyDrive/Ketastasia/models/{run_name}.h5'
    model.save(model_path)

    # 9. თვალყურის დევნება საუკეთესო ვალიდაციის მოდელზე
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_run_name = run_name

    print(f"✨ Run {run_name} დასრულდა. Val Accuracy: {val_acc:.4f}")
    wandb.finish()

print(f"\n{'='*60}\n🏆 Class Weights ექსპერიმენტები დასრულდა! საუკეთესო მოდელი ვალიდაციით: {best_run_name} (Val Acc: {best_val_acc:.4f})\n{'='*60}")


------------------------------------------------------------
🚀 იწყება Class Weights ექსპერიმენტი: lstm-h64-l1-lr0.01-bs32-classweights
------------------------------------------------------------


Epoch 1/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.2388 - loss: 2.6554 - val_accuracy: 0.2259 - val_loss: 2.6348 - learning_rate: 0.0100
Epoch 2/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.2881 - loss: 2.4247 - val_accuracy: 0.2724 - val_loss: 2.5190 - learning_rate: 0.0100
Epoch 3/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3408 - loss: 2.1797 - val_accuracy: 0.2121 - val_loss: 2.7356 - learning_rate: 0.0100
Epoch 4/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3541 - loss: 2.0795 - val_accuracy: 0.1638 - val_loss: 2.9741 - learning_rate: 0.0100
Epoch 5/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3895 - loss: 1.9055 - val_accuracy: 0.2966 - val_loss: 2.6122 - learning_rate: 0.0100
Epoch 6/40
89/92 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4401 - loss: 1.7170 
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4218 - loss: 1.7741 - val_accur

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✨ Run lstm-h64-l1-lr0.01-bs32-classweights დასრულდა. Val Accuracy: 0.4086


epoch/accuracy,▁▂▂▂▃▃▄▅▅▅▆▆▆▆▇▇▇▇▇▇██████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,██████▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▇▇▆▆▅▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▃▄▂▁▅▅▅▅▆▇▆▇▇▆▇▇▇▇██▇██▇██████
epoch/val_loss,▂▁▃▅▂▁▂▁▂▂▃▄▄▄▄▅▆▆▆▆▇▇▇▇▇█████
final_val_accuracy,▁
final_val_loss,▁
val_f1_macro,▁
val_f1_weighted,▁
epoch/accuracy,0.84286


...

Epoch 1/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 19s 151ms/step - accuracy: 0.2391 - loss: 2.7247 - val_accuracy: 0.1724 - val_loss: 2.8014 - learning_rate: 0.0010
Epoch 2/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 148ms/step - accuracy: 0.2949 - loss: 2.3436 - val_accuracy: 0.2500 - val_loss: 2.5736 - learning_rate: 0.0010
Epoch 3/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 161ms/step - accuracy: 0.3425 - loss: 2.2374 - val_accuracy: 0.2603 - val_loss: 2.4760 - learning_rate: 0.0010
Epoch 4/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 159ms/step - accuracy: 0.3745 - loss: 2.0472 - val_accuracy: 0.2138 - val_loss: 2.5185 - learning_rate: 0.0010
Epoch 5/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 20s 148ms/step - accuracy: 0.3779 - loss: 2.0072 - val_accuracy: 0.2172 - val_loss: 2.7702 - learning_rate: 0.0010
Epoch 6/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 22s 166ms/step - accuracy: 0.3714 - loss: 1.9778 - val_accuracy: 0.2914 - val_loss: 2.5264 - learning_rate: 0.0010
Epoch 7/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 155ms/step - accuracy: 0.4218 - loss: 1.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✨ Run lstm-h256-l2-lr0.001-bs32-classweights დასრულდა. Val Accuracy: 0.3966


epoch/accuracy,▁▂▂▃▃▃▃▄▄▄▄▅▆▆▆▇▇▇▇████
epoch/epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
epoch/learning_rate,███████████▄▄▄▄▂▂▂▂▁▁▁▁
epoch/loss,█▇▆▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁
epoch/val_accuracy,▁▃▄▂▂▅▅▆▄▅▆▆▇▆█▇█▆▇▇▇█▇
epoch/val_loss,▇▄▂▃▆▃▁▂▃▅▂▃▃▄▄▅▃▆▆▇█▇█
final_val_accuracy,▁
final_val_loss,▁
val_f1_macro,▁
val_f1_weighted,▁
epoch/accuracy,0.84048



🏆 Class Weights ექსპერიმენტები დასრულდა! საუკეთესო მოდელი ვალიდაციით: lstm-h64-l1-lr0.01-bs32-classweights (Val Acc: 0.4086)


both

In [18]:
# =====================================================================
# Cell 5 (V2): LSTM არქიტექტურა (Dropout-ის რეგულარიზაციით)
# =====================================================================
def build_lstm_with_dropout(hidden=64, n_layers=1, dropout_rate=0.3, lr=0.001):
    model = keras.Sequential()

    # შესავალი ფენა (15 კადრი, 29 ფიჩერი)
    model.add(layers.Input(shape=(n_timesteps, n_features)))

    # მრავალშრიანი LSTM დროპაუტებით
    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        model.add(layers.LSTM(
            hidden,
            return_sequences=return_seq,
            dropout=dropout_rate,           # 👈 დროპაუტი ინფუთებზე
            recurrent_dropout=dropout_rate  # 👈 დროპაუტი შიდა მეხსიერების კავშირებზე
        ))

    # გადაწყვეტილების მიმღები Dense ფენა
    model.add(layers.Dense(64, activation='relu'))

    # ცალკე დროპაუტი კლასიფიკატორის წინ
    model.add(layers.Dropout(dropout_rate))

    # გამომავალი შრე - Softmax კლასიფიკატორი
    model.add(layers.Dense(n_classes, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [19]:
# =====================================================================
# Cell 7: ექსპერიმენტის კონფიგურაციების განსაზღვრა (კომბინირებული ვერსია)
# =====================================================================
configs = [
    {
        'hidden': 64, 'n_layers': 1, 'dropout': 0.3,
        'lr': 0.01, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 64, 'n_layers': 1, 'dropout': 0.3,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 128, 'n_layers': 2, 'dropout': 0.4,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    },
    {
        'hidden': 128, 'n_layers': 2, 'dropout': 0.4,
        'lr': 0.0005, 'batch_size': 64, 'epochs': 40
    },
    {
        'hidden': 256, 'n_layers': 2, 'dropout': 0.3,
        'lr': 0.001, 'batch_size': 32, 'epochs': 40
    }
]
print(f"მომზადებულია {len(configs)} კომბინირებული (Dropout + Class Weights) ექსპერიმენტი! ⚖️🛡️")

მომზადებულია 5 კომბინირებული (Dropout + Class Weights) ექსპერიმენტი! ⚖️🛡️


In [20]:
# =====================================================================
# Cell 8: ექსპერიმენტების ავტომატური გაშვება (Dropout + Class Weights)
# =====================================================================
best_val_acc = 0
best_run_name = None

# საქაღალდე მოდელების შესანახად
os.makedirs('/content/drive/MyDrive/Ketastasia/models', exist_ok=True)

for cfg in configs:
    # სახელი თითოეული რანისთვის
    run_name = f"lstm-h{cfg['hidden']}-l{cfg['n_layers']}-d{cfg['dropout']}-lr{cfg['lr']}-bs{cfg['batch_size']}-combined"

    print(f"\n{'-'*60}\n🚀 იწყება კომბინირებული ექსპერიმენტი: {run_name}\n{'-'*60}")

    # 1. WandB-ს ინიციალიზაცია
    wandb.init(
        project='ildolcefarniente',
        group='p1_lstm',
        name=run_name,
        config={
            **cfg,
            'dropout_included': True,         # 👈 True
            'class_weights_included': True,   # 👈 True
            'n_timesteps': n_timesteps,
            'n_features': n_features,
            'n_classes': n_classes,
            'pipeline': 'pipeline1A_13kp',
            'normalization': 'per_frame_hip_centered'
        },
        reinit=True
    )

    # 2. მოდელის აწყობა -> ვიძახებთ დროპაუტიან ფუნქციას!
    model = build_lstm_with_dropout(
        hidden=cfg['hidden'],
        n_layers=cfg['n_layers'],
        dropout_rate=cfg['dropout'],
        lr=cfg['lr']
    )

    # 3. ქოლბექები
    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=4,
            verbose=1
        )
    ]

    # 4. მოდელის წვრთნა -> ორივე ერთად მუშაობს (წონებიც და არქიტექტურაში დროპაუტიც!)
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        class_weight=class_weight_dict,       # 👈 კლასების წონები
        callbacks=callbacks,
        verbose=1
    )

    # 5. საბოლოო შეფასება (Validation)
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)
    y_true_val = np.argmax(y_val, axis=1)

    # კლასიფიკაციის რეპორტი
    report_val = classification_report(
        y_true_val,
        y_pred_val,
        labels=list(range(len(classes))),
        target_names=classes,
        output_dict=True
    )

    # 6. Confusion Matrix გრაფიკის აგება
    cm_val = confusion_matrix(y_true_val, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Val Confusion Matrix (Combined) — {run_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Val Label')
    ax.set_xlabel('Predicted Val Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'/content/{run_name}_val_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    # 7. ფინალური მეტრიკების ატვირთვა WandB-ზე
    wandb.log({
        'final_val_accuracy':        val_acc,
        'final_val_loss':            val_loss,
        'val_f1_macro':              report_val['macro avg']['f1-score'],
        'val_f1_weighted':           report_val['weighted avg']['f1-score'],
        'val_confusion_matrix_db':   wandb.plot.confusion_matrix(
            probs=None, y_true=y_true_val.tolist(), preds=y_pred_val.tolist(), class_names=classes
        ),
        'val_confusion_matrix_img':  wandb.Image(cm_path)
    })

    # 8. მოდელის შენახვა Drive-ზე
    model_path = f'/content/drive/MyDrive/Ketastasia/models/{run_name}.h5'
    model.save(model_path)

    # 9. საუკეთესო მოდელის ფიქსაცია
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_run_name = run_name

    print(f"✨ Run {run_name} დასრულდა. Val Accuracy: {val_acc:.4f}")
    wandb.finish()

print(f"\n{'='*60}\n🏆 ყველა ფაზის ექსპერიმენტი მორჩა! საუკეთესო კომბინირებული მოდელი: {best_run_name} (Val Acc: {best_val_acc:.4f})\n{'='*60}")


------------------------------------------------------------
🚀 იწყება კომბინირებული ექსპერიმენტი: lstm-h64-l1-d0.3-lr0.01-bs32-combined
------------------------------------------------------------


Epoch 1/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - accuracy: 0.1303 - loss: 3.0964 - val_accuracy: 0.2621 - val_loss: 2.6637 - learning_rate: 0.0100
Epoch 2/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.1789 - loss: 2.8386 - val_accuracy: 0.2086 - val_loss: 2.6606 - learning_rate: 0.0100
Epoch 3/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.2333 - loss: 2.6666 - val_accuracy: 0.1914 - val_loss: 2.6994 - learning_rate: 0.0100
Epoch 4/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.1952 - loss: 2.7627 - val_accuracy: 0.1828 - val_loss: 2.7387 - learning_rate: 0.0100
Epoch 5/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.2282 - loss: 2.5704 - val_accuracy: 0.1879 - val_loss: 2.6421 - learning_rate: 0.0100
Epoch 6/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.2582 - loss: 2.4479 - val_accuracy: 0.2310 - val_loss: 2.6625 - learning_rate: 0.0100
Epoch 7/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.2514 - loss: 2.4582 - val_acc

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

✨ Run lstm-h64-l1-d0.3-lr0.01-bs32-combined დასრულდა. Val Accuracy: 0.2621


epoch/accuracy,▁▃▆▄▆▇▇██
epoch/epoch,▁▂▃▄▅▅▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▄▅▃▂▃▂▁
epoch/val_accuracy,█▃▂▁▁▅▄▄▁
epoch/val_loss,▅▅▇█▅▅▂▂▁
final_val_accuracy,▁
final_val_loss,▁
val_f1_macro,▁
val_f1_weighted,▁
epoch/accuracy,0.27687


...

Epoch 1/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 41s 347ms/step - accuracy: 0.1327 - loss: 3.0294 - val_accuracy: 0.1707 - val_loss: 2.8526 - learning_rate: 0.0010
Epoch 2/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 31s 333ms/step - accuracy: 0.1554 - loss: 2.8533 - val_accuracy: 0.1207 - val_loss: 2.7462 - learning_rate: 0.0010
Epoch 3/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 41s 328ms/step - accuracy: 0.1759 - loss: 2.8425 - val_accuracy: 0.2121 - val_loss: 2.6761 - learning_rate: 0.0010
Epoch 4/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 32s 343ms/step - accuracy: 0.1680 - loss: 2.7209 - val_accuracy: 0.1931 - val_loss: 2.7409 - learning_rate: 0.0010
Epoch 5/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 31s 338ms/step - accuracy: 0.1888 - loss: 2.6731 - val_accuracy: 0.2138 - val_loss: 2.6141 - learning_rate: 0.0010
Epoch 6/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 31s 338ms/step - accuracy: 0.2177 - loss: 2.6446 - val_accuracy: 0.1828 - val_loss: 2.6881 - learning_rate: 0.0010
Epoch 7/40
92/92 ━━━━━━━━━━━━━━━━━━━━ 31s 340ms/step - accuracy: 0.2282 - loss: 2.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✨ Run lstm-h256-l2-d0.3-lr0.001-bs32-combined დასრულდა. Val Accuracy: 0.3069


epoch/accuracy,▁▂▂▂▂▃▃▃▅▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇█████
epoch/epoch,▁▁▁▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇██
epoch/learning_rate,█████████████████████▃▃▃▃▁▁▁▁
epoch/loss,█▇▇▆▆▆▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▁▁▁▁▁
epoch/val_accuracy,▃▁▄▄▅▃▆▄▅▄▆▆▅▅▅▇▆▇▇▇█▆▇████▇█
epoch/val_loss,█▆▅▆▄▆▃▄▂▃▃▂▂▃▂▁▁▃▂▃▂▁▂▃▁▁▂▂▂
final_val_accuracy,▁
final_val_loss,▁
val_f1_macro,▁
val_f1_weighted,▁
epoch/accuracy,0.41531



🏆 ყველა ფაზის ექსპერიმენტი მორჩა! საუკეთესო კომბინირებული მოდელი: lstm-h256-l2-d0.3-lr0.001-bs32-combined (Val Acc: 0.3069)


In [21]:
# =====================================================================
# Cell 9: საუკეთესო LSTM მოდელის ავტომატური შენახვა Drive-ზე და WandB-ზე
# =====================================================================
import os
import wandb

# 1. განვსაზღვროთ ჩვენი გამარჯვებული მოდელის მონაცემები
best_run_name = "lstm-h256-l2-lr0.001-bs32-baseline"
best_val_f1_macro = 0.3153  # შენი მოდელის ზუსტი F1 ქულა

print(f"საუკეთესო შერჩეული მოდელი: {best_run_name}")
print(f"Validation F1-Macro: {best_val_f1_macro:.4f}")

# 2. ვქმნით საქაღალდეს Google Drive-ზე
models_dir = '/content/drive/MyDrive/Ketastasia/models/'
os.makedirs(models_dir, exist_ok=True)

drive_model_path = os.path.join(models_dir, 'pipeline1A_best_lstm.h5')

if 'model' in locals():
    model.save(drive_model_path)
    print(f" მოდელი წარმატებით შეინახა Google Drive-ზე: {drive_model_path}")
else:
    print(" ყურადღება: 'model' ცვლადი ვერ მოიძებნა მეხსიერებაში!")


run = wandb.init(
    project="ildolcefarniente",
    job_type="model_registration",
    name="register_pipeline1A_lstm"
)

artifact = wandb.Artifact(
    name="pipeline1A_lstm_models",
    type="model",
    description=f"Best 2-layer LSTM model with 256 hidden units. Validation F1-Macro: {best_val_f1_macro:.4f}."
)


artifact.add_file(drive_model_path)


run.log_artifact(artifact, aliases=["latest"])
run.finish()

print("\n მორჩა, ყველაფერი სინქრონიზებულია და მზადაა! 🔥")

საუკეთესო შერჩეული მოდელი: lstm-h256-l2-lr0.001-bs32-baseline
Validation F1-Macro: 0.3153
 მოდელი წარმატებით შეინახა Google Drive-ზე: /content/drive/MyDrive/Ketastasia/models/pipeline1A_best_lstm.h5



 მორჩა, ყველაფერი სინქრონიზებულია და მზადაა! 🔥
